# Notebook 4 — Duplicate Data Handling
### Sprint 5 | Data Cleaning & Preprocessing for AI/ML Engineers


In [1]:
import pandas as pd
df = pd.read_csv("telco_churn.csv")
print(f"Dataset loaded: {df.shape[0]:,} rows")


Dataset loaded: 7,043 rows


---
## 1. What Are Duplicate Records? 2. Exact vs Partial Duplicates

### Understand
An **exact duplicate** is a row that's identical to another in *every* column, including
any identifier — a true, unambiguous repeat. A **partial duplicate** matches on some
columns (often all except an identifier) but differs elsewhere, or matches on every
column except the ID — this is a much more ambiguous case that needs judgment, not an
automatic rule.

### Demonstrate
**Business example:** Two rows with the exact same `customerID` would be an exact
duplicate — clearly an error. Two rows with *different* `customerID`s but identical
demographic and billing profiles are a partial duplicate — possibly two different real
customers who happen to share a profile, not an error at all.

### Implement


In [2]:
exact_duplicates = df.duplicated().sum()
print(f"Exact duplicate rows (every column, including customerID): {exact_duplicates}")

partial_duplicates = df.drop(columns=['customerID']).duplicated(keep=False).sum()
print(f"Rows involved in a PARTIAL duplicate group (same profile, different customerID): {partial_duplicates}")


Exact duplicate rows (every column, including customerID): 0
Rows involved in a PARTIAL duplicate group (same profile, different customerID): 42


**Finding:** Zero exact duplicates — no true error-type repeats exist. But 42 rows
(21 pairs/groups) share an identical profile once `customerID` is excluded — this is the
ambiguous case this whole notebook is really about.


---
## 3. Identifying Duplicates — `duplicated()`

### Understand
`.duplicated()` returns a boolean Series flagging every row that's a repeat of an
*earlier* row — the first occurrence is `False`, later matches are `True`.

### Implement


In [3]:
dup_flags = df.duplicated()
print(f"Rows flagged as duplicates: {dup_flags.sum()}")
print(f"(Compare to keep=False, which flags ALL rows in a duplicate group, not just later ones)")
print(f"Rows flagged with keep=False: {df.duplicated(keep=False).sum()}")


Rows flagged as duplicates: 0
(Compare to keep=False, which flags ALL rows in a duplicate group, not just later ones)
Rows flagged with keep=False: 0


**Finding:** Both counts are 0 for full-row duplicates — confirming `customerID`
alone is sufficient to make every row unique.


---
## 4. `drop_duplicates()`

### Understand
`.drop_duplicates()` removes duplicate rows, keeping the first occurrence by default.

### Implement


In [4]:
before = len(df)
deduped = df.drop_duplicates()
print(f"Rows before: {before} | Rows after drop_duplicates(): {len(deduped)} | Removed: {before - len(deduped)}")


Rows before: 7043 | Rows after drop_duplicates(): 7043 | Removed: 0


**Finding:** No rows removed — there was nothing to remove at the full-row level.


---
## 5. Duplicate Detection Based on Selected Columns

### Understand
Checking duplicates on a *subset* of columns (e.g., excluding the identifier) reveals
partial duplicates that a full-row check misses entirely.

### Implement


In [5]:
subset_dupes = df.drop(columns=['customerID']).duplicated(keep=False)
print(f"Rows sharing an identical profile (excluding customerID): {subset_dupes.sum()}")

example_group = df[subset_dupes].sort_values(
    by=[c for c in df.columns if c != 'customerID']
).head(4)
print("\nExample duplicate group (different customerID, identical everything else):")
print(example_group[['customerID', 'gender', 'tenure', 'MonthlyCharges', 'Churn']])


Rows sharing an identical profile (excluding customerID): 42

Example duplicate group (different customerID, identical everything else):
      customerID  gender  tenure  MonthlyCharges Churn
6491  9728-FTTVZ  Female       1            69.2   Yes
6764  7660-HDPJV  Female       1            69.2   Yes
4495  4702-IOQDC  Female       1            70.1   Yes
6267  0328-GRPMV  Female       1            70.1   Yes


**Finding:** 42 rows fall into 21 such groups. This is a *much* more consequential
finding than the exact-duplicate check — and exactly the kind of result that needs
careful judgment, not an automatic `drop_duplicates()` call, which is this notebook's
central lesson.


---
## 6. Handling Duplicate Records & 7. Business Rules for Duplicate Removal

### Understand
The right handling decision depends entirely on what a "duplicate" *means* for this
specific business — not a generic rule. For Telco customer data specifically: could two
different real customers plausibly share an identical demographic and billing profile by
coincidence? With `tenure=1`, `MonthlyCharges` rounded to the cent, and matching service
selections across 16 categorical columns — yes, entirely plausible in a company with
thousands of customers and a limited set of standard pricing plans.

### Demonstrate — Why Blindly Deleting These Would Be Dangerous


In [6]:
# If these were blindly treated as duplicates and removed...
naive_dedup = df.drop(columns=['customerID']).drop_duplicates()
print(f"Naive dedup (ignoring customerID) would reduce the dataset from {len(df)} to {len(naive_dedup)} rows")
print(f"That's {len(df) - len(naive_dedup)} REAL, uniquely-identified customers that would be silently deleted.")


Naive dedup (ignoring customerID) would reduce the dataset from 7043 to 7021 rows
That's 22 REAL, uniquely-identified customers that would be silently deleted.


**Business Rule Applied:** Because `customerID` is a genuine, independently
verified unique identifier (confirmed in Sprint 4, Notebook 1 — 100% unique, no
duplicates), and no two customers share an ID, **every row in this dataset represents a
real, distinct customer**, regardless of how similar their profiles look. The correct
business rule is: **duplicates are defined by identifier match, not by attribute
similarity, for this dataset.** Two customers legitimately CAN have identical service
plans and billing amounts — that's not an error, it's two people who happen to have made
the same choices.

### Documentation (Problem / Analysis / Technique / Reason / Implementation / Result / Impact)
- **Problem:** 42 rows share an identical profile once `customerID` is excluded.
- **Analysis:** Each has a distinct, verified-unique `customerID` — these are different
  customers, not database repeats.
- **Technique Selected:** No removal. Duplicate status is judged by `customerID`, not by
  attribute similarity.
- **Reason:** Removing these rows would delete real customers based on a coincidence
  (shared plan choices), not an actual data error — exactly the danger this notebook
  warns against.
- **Implementation:** `df.duplicated()` (full-row, including `customerID`) is the correct
  check for this dataset; the subset check (Topic 5) is diagnostic only, not
  action-triggering.
- **Result:** 0 rows removed. Dataset remains at 7,043 rows.
- **Impact:** Preserves every real customer record, avoiding an artificial reduction in
  training data and avoiding bias against whatever common profile combination happened to
  be shared.


---
## Summary

| Check | Result | Action Taken |
|---|---|---|
| Exact duplicates (full row incl. ID) | 0 | None needed |
| Partial duplicates (excl. customerID) | 42 rows / 21 groups | **Investigated, NOT removed** — verified as distinct real customers |

**Central lesson demonstrated with a real example, not a hypothetical:** a duplicate
*detection* rule (matching attributes) and a duplicate *removal* rule (business meaning of
"the same record") are not the same thing — this dataset had zero database-level errors,
despite having a nontrivial number of coincidentally-identical customer profiles.

**Next notebook:** `05_Data_Validation.ipynb` — building explicit validation rules (range,
type, format, category, business-rule) as Python functions.
